# Gauss-Newton Matrix

In [9]:
import numpy as np
from scipy.spatial.transform import Rotation as R

## 1. COE2RV

NOTE: Need to use semiparameter ($p$) instead of semimajor axis ($a$). $a$ is infinite for the parabola, whereas $p$ is defined for all orbits

In [10]:
def COE2RV(coe, mu=3.986004418*(10**14)):

    # INPUT: 
    #   coe is an array of the Keplerian Orbital Elements
    #       a - semi-major axis
    #       e - eccentricity
    #       i - inclination
    #       node - right ascension of the ascending node
    #       arg - argument of perigee
    #       nu - true anomaly
    #   mu - gravitational parameters (=GM). Default set to the value for Earth

    # OUTPUT:
    #   r - position vector of satellite
    #   v - velocity vector of satellite


    a, e, i, node, arg, nu = coe

    sin_nu = np.sin(np.deg2rad(nu))
    cos_nu = np.cos(np.deg2rad(nu))

    # Calculate semiparameter (p)
    p = a * (1-e**2)

    # Perifocal Coordinate System
    R_PQW = np.zeros(3)
    V_PQW = np.zeros(3)

    R_PQW[0] = (p * cos_nu) / (1 + e*cos_nu)
    R_PQW[1] = (p * sin_nu) / (1 + e*cos_nu)
    
    V_PQW[0] = -1 * np.sqrt(mu/p) * sin_nu
    V_PQW[1] = np.sqrt(mu/p) * (e + cos_nu)

    # Rotation Matrix
    R_node_z = R.from_euler('z', np.deg2rad(node)).as_matrix()
    R_i_x    = R.from_euler('x', np.deg2rad(i)).as_matrix()
    R_arg_z  = R.from_euler('z', np.deg2rad(arg)).as_matrix()
    R_total  = R_node_z @ R_i_x @ R_arg_z

    # Rotate Perifocal to IJK
    R_IJK = R_total @ R_PQW
    V_IJK = R_total @ V_PQW

    return R_IJK, V_IJK

### Example 2.6 from FoAaA (pg. 119)

Verifying function is correct

In [15]:
p = 11067.790 #km
e = 0.83285
i = 87.87
node = 227.89
arg = 53.38
nu = 92.335

a = (p*1000) / (1-e**2)

coe = [a, e, i, node, arg, nu]

r, v = COE2RV(coe)

print("Vector r (m):")
print("Textbook: [6525344  6861535  6449125]")
print("Mine:    ", r)

print("\nVector v (m/s):")
print("Textbook: [4902.276  5533.124  -1975.709]")
print("Mine:    ", v)

Vector r (m):
Textbook: [6525344  6861535  6449125]
Mine:     [6525368.12098609 6861531.83489605 6449118.61416016]

Vector v (m/s):
Textbook: [4902.276  5533.124  -1975.709]
Mine:     [ 4902.27864642  5533.13956836 -1975.71009954]


## Doppler Shift from $\textbf{r}$ and $\textbf{v}$

In [ ]:
def f_D(r, v, r_gs, v_gs, f_c, c=299792458):
    rho = r - r_gs
    rho_hat = rho / np.sqrt(rho.dot(rho))

    v_rel = v - v_gs

    k = f_c/c

    f_D = k * rho_hat * (-1 * v_rel)

    return f_D